### HRSMC Course: Applied machine learning for chemistry
# Day 2, Introduction to Machine Learning
### by Bernd Ensing, University of Amsterdam

The aim of this first Jupyter Notebook of Day 2, we will go through an exercise of supervised learning applied to one-dimensional regression. In this notebook, you will be asked a few questions, but you do not need to code any python yet. However, please do carefully inspect the jupyter codes every time, and check if you can understand the flow of the algorithms. In the second Jupyter Notebook, you will apply what you learned here and write some python code yourself. This first notebook should take you about 30 minutes.

## Assignment 1 - Simple regression

In this notebook, you will optimize a fit function through a set of data points representing a time correlation function. For this exercise, the dataset is generated from a simple function, so that we actually know the target "ground truth" function underlying the data. To resemble realistic measurement data, we add some random noise.

The fit functions are polynomials of the form:

$$
y=\sum_{i=0}^{N} c_i*x^i
$$

in which $N$ sets the polynomial degree (i.e. the complexity of our model) and $\{c_i\}$ are the coefficients to be optimized. Although the polynomial is of course non-linear for $N>1$, it is actually a linear function of the coefficients.

Tasks to fulfill:
1. generate and examine the data
1. optimize the coefficients of the polynomial using a numpy solver
1. divide the data in a training and a validation set
1. define a loss function
1. assess fit error versus generalization performance
1. implement your own regression optimization function
1. apply feature scaling
1. implement and evaluate L1 and L2 regularization
1. assess how the optimal polynomial order changes when more training data is available
1. OPTIONAL: K-fold cross validation
1. OPTIONAL: using a one-layer neural network as fit function



In [ ]:
# First load some useful libraries
import numpy as np
import math
import matplotlib.pyplot as plt

## Data generation

We will generate data points from an analytical (ground truth) function with the following shape:

$$f(x) = \cos(4x) \cdot e^{-x}$$

This function resembles a typical normalized auto-correlation function (ACF), for example of the velocity of a particle in a solvent.

Let's write below a python function, <code>my_acf(x)</code> that takes $x$ as input and returns $f(x)$. Use matplotlib to plot the function from <code>xmin = 0</code> to <code>xmax = 3.0</code>.

In [ ]:
# my auto correlation function
def my_acf[T: float | np.typing.NDArray[np.floating]](x: T) -> T:
    return np.cos(4.0*x) * np.exp(-x) # type: ignore

We fill two numpy arrays, <code>xacf</code> and <code>yacf</code>, with <code>nacf = 1000</code> points on the range of xmin and xmax and plot the result.

In [ ]:
# fill x and y arrays
nacf = 1000
xmin = 0.0
xmax = 3.0
xacf = np.linspace(xmin, xmax, nacf)
yacf = my_acf(xacf)

# plot ground truth function with matplotlib
plt.figure(figsize=(6, 5))
plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(np.arange(min(xacf), max(xacf)+1.0, 1.0),fontsize=16)
plt.yticks(([-0.5, 0.0, 0.5, 1.0]),fontsize=16)
plt.show()


Now, let's generate <code>ndata=24</code> data points on this $x=[0,3]$ range. To make the data in this exercise resemble realistic data from a noisy measurement, we add random noise to our data points. Use numpys <code>np.random.randn()</code> to draw random numbers from a Gaussian distribution, $\mathcal{N}(\mu,\sigma^2)$, centered at $\mu = 0$ and a width of $\sigma=0.1$. Add this to each data point.

We plot again the result together with our ground truth function.

In [ ]:
# Let's set the random number seed for reproducibility
np.random.seed(123)

# Create the data points
ndata = 24
scale = 0.1
xdata = np.linspace(xmin, xmax, ndata)
ydata = my_acf(xdata)
for i in range(ndata):
    ydata[i] += scale * np.random.randn()


# plot ground truth function togethert with data points with matplotlib
plt.figure(figsize=(6, 5))
plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.scatter(xdata, ydata,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='noisy data set')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(np.arange(min(xacf), max(xacf)+1.0, 1.0),fontsize=16)
plt.yticks(([-0.5, 0.0, 0.5, 1.0]),fontsize=16)
plt.show()


## Curve fitting using a polynomial function

Curve fitting is easy; there are various software packages that can do the job for you. To fit a polynomial function through our data points, we can use for example numpys [polyfit()](https://numpy.org/doc/stable/reference/generated/numpy.polynomial.polynomial.polyfit.html) function.

* use first <code>polyfit()</code> to fit a polynomial of degree $N=3$ to the data points, and obtain the array of coefficients, which is returned by the function
* use <code>polyval()</code> to compute the polynomial function for the <code>nacf = 1000</code> xacf values.
* plot the result together with the ground truth function and the data points for comparison

In [ ]:
import numpy.polynomial.polynomial as poly
# fit a polynomial through the noisy data point and store the result in array yfit
ndegree=6
coefs = poly.polyfit(xdata, ydata, ndegree)
yfit = poly.polyval(xacf, coefs)

# plot the result
plt.figure(figsize=(6, 5))
plt.title("Polynomial fit of degree "+str(ndegree))
plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.scatter(xdata, ydata,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='noisy data set')
plt.plot(xacf, yfit, color="red",linewidth=2.5,label='polynomial fit')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(np.arange(min(xacf), max(xacf)+1.0, 1.0),fontsize=16)
plt.yticks(([-0.5, 0.0, 0.5, 1.0]),fontsize=16)
plt.ylim([-0.75, 1.2])
plt.show()

# print the coefficients
print(f"  Polynomial coefficients {', '.join(str(round(c, 3)) for c in coefs)}")

### Which polynomial degree?

Clearly, a polynome of degree 3 does not give a good fit. Perhaps you already tried to run the above cell with different values of <code>ndegree</code>. If not, go ahead and try some larger polynomial degrees to see how the fit improves.

$\color{DarkBlue}{\textbf{Question 1}}$
* Which polynomial degree do you recommend to fit the data points? Why? 

Answer: I think degree 6 or 7 is the best choice:
- 6 represents the data between 0 and 2 well, but its shape/fit is not very good around 2.8.
- 7 approaches the data points around 2.5 better, but it may be already overfitting the data.


To compare the different polynomial fit functions, let's run the curve fitting for a range of degree values and plot them next to each other:

In [ ]:
fig, axs = plt.subplots(nrows=4, ncols=2, sharey=False, figsize=(12, 16))
axs = axs.flatten()                         # let's keep the numbering of the sub figures 1-dimensional
degree_list = ([1, 2, 3, 4, 6, 8, 12, 18])  # list of polynomial degree values to compute and plot
for i, ndegree in enumerate(degree_list):
    ax = axs[i]

    #  compute again the polynomial with polyfit() and polyval()
    #  create again the figures using ax.plot() or ax.scatter()
    coefs = poly.polyfit(xdata, ydata, ndegree)
    yfit = poly.polyval(xacf, coefs)

    ax.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
    ax.scatter(xdata, ydata,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='noisy data set')
    ax.plot(xacf, yfit, color="red",linewidth=2.5,label='polynomial fit')
    print(f"N = {ndegree}, C= {', '.join(str(round(c, 3)) for c in coefs)}")

    ax.set_ylim([-0.75, 1.2])
    ax.set_title(f"Polynomial fit of degree {str(ndegree)}")
    if i > 5:
        ax.set_xlabel('x',fontsize=18)
    if i % 2 == 0:
        ax.set_ylabel('y',fontsize=18)
plt.tight_layout()
plt.show()

## Conclusion part 1

Congratulations! You have fitted many polynomial functions of different degrees to the noisy data!
This concludes the first part of this exercise on simple regression.

Well, perhaps the curve fitting results are still somewhat unsatisfactory in a certain respect. In particular, we have not discussed which polynomial fit is optimal. How should we assess this? Is it enough to just measure the accuracy of the fit defined as the sum of the discrepancies of the polynomial with respect to the data points?

Clearly, the polynomial functions of a small degree perform badly. The functions do not pass through any of the data points and they do not resemble the ground truth function. Having too few degrees of freedom (i.e. coefficients) and relatively simple terms ($x$, $x^2$, $x^3$,...) results in severe <i>underfitting</i> of the data. The model (i.e. the polynomial) is said to be not <i>expressive</i> enough.

On the other hand, the polynomial of very high degree does an excellent job in going through most data points. However, the result is a very wiggly function that does not resemble the ground truth function, as it tries to fit the noise! The result is a classical case of <i>overfitting</i>.

A tell-tale characteristic in such overfitting can be seen in the values of the fitted coefficients.
* Go back to the previous cell above, and add a print statement of the coefficients inside the for-loop to see the coefficients for the different polynomial degrees.

Notice anything suspicious?
At the highest polynomial degree, which suffer from overfitting, the coefficients take extremely large values with alternating negative and positive signs. The regression task is over-determined by a too expressive function, in which the subsequent polynomial terms are trying to compensate for each other.

# The machine learning approach

In the next part, we will divide our data into two parts, a training set and a validation set. The training set is used for training the model (here, fitting the polynomial). The validation set is used to assess how well the model generalizes to new data that was "not seen" during the training phase. To evaluate the performance, we will use an error, or loss, function.

Secondly, we will implement our own curve fitting function. We will not use the most optimal approach to determine the coefficients that optimize the fit to the data, but instead will use a "steepest descent" algorithm that is very illustrative to the working of a broad range of machine learning approaches. Secondly, we will be able to augment this algorithm with several machine learning concepts that allow us to deal with the problem of overfitting using <i>regularization</i>.



Hereafter, we will discuss and implement the following concepts: 
* training, validation (and testing)
* loss function
* learning rate
* feature scaling
* regularization
* performance dependence on the amount of data


## Splitting the data in training and validation sets

Let's start by dividing our data set (<code>xdata, ydata</code>) of 24 points into a training set (<code>xtrain, ytrain</code>) and a validation set (<code>xval, yval</code>). We will split the data into two thirds for training and one third for validation. Make sure that both sets are representative over the whole range. 

$\color{DarkBlue}{\textbf{Question 2}}$
* In machine learning tasks, one actually often divides the data into three parts: a training set, a validation set, and a test set. What could be the purpose of this third test set?

The test set could be used to expand the scope of the model, i.e., te see whether it can describe data that is not in the same range as the training set nor the validation set.

$\color{DarkBlue}{\textbf{Question 3}}$
* Is it wise to divide the data randomly over training and validation sets? Why?

The whole point is that the training set should be representative of the whole data set. It is wise to divide the data randomly over the training and validation sets if both sets are large enough (which makes them representative). However, store the random seed to make the results reproducible.

Let's go ahead and split the data.

In [ ]:
# divide the data points into a training set and a validation set
ntrain = 2 * ndata // 3 + (ndata % 3 > 0)
nval = ndata // 3 + (ndata % 3 == 2)
xtrain = np.zeros(ntrain)
ytrain = np.zeros(ntrain)
xval = np.zeros(nval)
yval = np.zeros(nval)
itrain = ival = 0
print(ndata,ntrain,nval)
for idat in range(ndata):
    if idat % 3 == 1:
        xval[ival] = xdata[idat]
        yval[ival] = ydata[idat]
        ival += 1
    else:
        xtrain[itrain] = xdata[idat]
        ytrain[itrain] = ydata[idat]
        itrain += 1


In [ ]:
# let's plot the training and validation data as a visual check
plt.figure(figsize=(6, 5))
plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.scatter(xdata, ydata,marker="x",facecolors='y',s=80,linewidth=2.5,label='noisy data set')
plt.scatter(xtrain, ytrain,marker="o",edgecolors='b',facecolors='None',s=80,linewidth=2,label='training set')
plt.scatter(xval, yval,edgecolors='r',facecolors='None',s=80,linewidth=2,label='validation set')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(np.arange(min(xtrain), max(xtrain)+1.0, 1.0),fontsize=16)
plt.yticks(([-0.5, 0.0, 0.5, 1.0]),fontsize=16)
plt.show()

### Loss function

To measure the fitting error, we will need a loss function. Note that also inside the numpy polynomial fitting algorithm a similar loss function is used as a minimization target. We will use the mean square error (MSE) as the loss function:
$$ \mathrm{MSE} = \frac{1}{N} \sum_{i=0}^N (y^\mathrm{pred}_i - y^\mathrm{target}_i)^2 $$

Here $N$ is the number of training or validation data points, $y^\mathrm{target}$ are the target $y$-values and $y^\mathrm{pred}$ are the predictions from the model.

Let's write a function <code>my_loss()</code> that takes as input the target and prediction values, and returns the MSE.

In [ ]:
# MSE loss function
def my_loss(y,t):
    return np.square(y - t).sum() / len(y)

Next, we will write the routine to compute and plot the training loss and the validation loss as a function of the polynomial degree.

In [ ]:
# declare some arrays
maxdegree = 16
ypred_train = np.empty(ntrain)
ypred_val = np.empty(nval)
loss_train = np.empty(maxdegree)
loss_val = np.empty(maxdegree)

# for polynomial degrees zero to maxdegree:
#    fit the polynomial
#    compute the predicted values for the training and validation sets
#    compute the training and validation loss
for ndegree in range(maxdegree):

    coefs = poly.polyfit(xtrain, ytrain, ndegree)

    ypred_train = poly.polyval(xtrain, coefs)
    ypred_val = poly.polyval(xval, coefs)

    loss_train[ndegree] = my_loss(ypred_train, ytrain)
    loss_val[ndegree] = my_loss(ypred_val, yval)

# print the training and validation loss
print(" degree    training loss   validation loss")
for ndegree in range(maxdegree):
    print(f"     {ndegree:3d}     {loss_train[ndegree]:8.4f}    {loss_val[ndegree]:8.4f}")


# plot the training and validation loss as a function of polynomial degree
xdegree = np.arange(maxdegree)
plt.figure(figsize=(6, 5))
plt.scatter(xdegree, loss_train,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='')
plt.scatter(xdegree, loss_val,edgecolors='r',facecolors='None',s=80,linewidth=2.5,label='')
plt.plot(xdegree, loss_train, 'b-',linewidth=2.5, label='Training loss')
plt.plot(xdegree, loss_val, 'r-',linewidth=2.5, label='Validation loss')
plt.xlabel('polynomial order',fontsize=20)
plt.ylabel('loss',fontsize=20)
plt.xticks(np.arange(0,maxdegree, 2.0),fontsize=16)
plt.yticks(fontsize=16)
plt.ylim([-0.05, 0.2])
plt.legend(fontsize=16)

$\color{DarkBlue}{\textbf{Question 4}}$
* Based on the computed training and validation errors, what is the optimal polynomial degree to fit the data? Why?

Between 7 and 8 because then the validation loss is at a minumum. After this step, the validation loss sarts to increase drastically, which is a sign of overfitting.

## Implementing a curve fitting algorithm

The curve fitting algorithm is an iterative process that optimizes the coefficients with the aim to minimize the loss function. The main loop contains two main steps:
* forward pass: compute the model prediction (i.e. polynomial y-values) and the loss
* backward pass: compute the derivatives of the loss with respect to the coefficients and update the coefficients

In neural network optimizers (which we will see later in the course), the second step is called back-propagation, which is why we refer to this step as the "backward pass" here. The reason behind this nomenclature has to do with the direction of the application of the chain-rule to compute the derivatives, starting from the loss. In our case:

$$\frac{\partial \mathcal{L}}{\partial c_i} = \sum_{j=0}^N \frac{\partial \mathcal{L}}{\partial y^\mathrm{pred}_j} \frac{\partial y^\mathrm{pred}_j}{\partial c_i}$$

* $\mathcal{L}$ is the loss function, for which we can use the squared error (why can we ommit the factor $1/N$ here?)
* the sum runs over all data points

As a side note: in machine learning tasks often the data sets are very large, so that summing over all data points for each coefficient becomes computationally expensive. The trick is then to use only a small, randomly chosen, batch of data points for each iteration. This approach is called <i>stochastic gradient descent</i> and is the standard optimization algorithm for neural networks, as we will see later.


$\color{DarkBlue}{\textbf{Question 5}}$

Give the expression for the $\frac{\partial \mathcal{L}}{\partial y}$ term.

$\color{Grey}{\textsf{<double-click and type your answer here>}}$

$\color{DarkBlue}{\textbf{Question 6}}$

Give the expression for the $\frac{\partial y}{\partial c}$ term.

$\color{Grey}{\textsf{<double-click and type your answer here>}}$

Can you understand all the steps in the algorithm code below?

In [ ]:
# set some parameters
ndegree = 3
learning_rate = 1e-4
maxiter = 1e6
toll = 1e-7
nprint = 10000  # print interval
ndegree += 1    # to include the last item in loops and declarations

# declare arrays
coef = np.zeros(ndegree)
grad_coef = np.zeros(ndegree)
y_pred = np.zeros(xtrain.size)

# initialize
iter = 0
converged = False
loss = 99999.
print("Fitting polynomial of degree: ",ndegree-1)
print("#  iter      loss    improvement")

# main optimization loop
while iter < maxiter and not converged:
    # Forward pass: compute y_pred = c0 + c1 x + c2 x^2 + c3 x^3 + ...
    y_pred[:] = coef[0]
    for i in range(1,ndegree):
        y_pred += coef[i] * xtrain**i

    loss_prev = loss
    # Compute loss
    loss = np.square(y_pred - ytrain).sum()

    # print progress
    if iter % nprint == 0:
        print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))
    # check convergence
    if( abs(loss - loss_prev) < toll ):
        converged = True

    # Backprop to compute gradients of coeficients with respect to loss
    grad_y_pred = 2.0 * (y_pred - ytrain)
    for i in range(ndegree ):
        grad_coef[i] = (grad_y_pred * xtrain**i).sum()

    # Update weights
    coef -= learning_rate * grad_coef

    iter += 1

# print progress final epoch
print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))


# compute polynomial function and plot result together with training points and ground truth
yfit= np.zeros(xacf.size)
yfit[:] = coef[0]
for i in range(1,ndegree):
    yfit += coef[i] * xacf**i
plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.scatter(xtrain, ytrain,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='noisy data set')
plt.plot(xacf, yfit, color="red",linewidth=2.5,label='polynomial fit')
plt.legend()
plt.show()

# print polynomial expansion to screen
line = 'Result: y = ' + str('{:8.4f}'.format(coef[0]))
for i in range(1,ndegree):
    line += ' + ' + str('{:8.4f}'.format(coef[i])) + ' * x^' + str(i)
print(line)



### Learning rate

You probably notice that our optimization algorithm is significantly slower than the numpy polyfit function. One reason is that for linear problems, there are much faster algorithms than steepest descent. Secondly, the numpy uses compiled and highly optimized C/C++ routines, which run very efficiently. 

An important parameter for the performance of our steepest descent implementation is the learning rate. A too small learning rate will requires more cycles, whereas a too large learning rate causes instabilities and divergence.

Run your algorithm in the previous cell for higher polynomial degrees and try to find good values for the parameters <code>learning_rate</code>, <code>maxiter</code>, and <code>toll</code> to arrive at solutions that are comparable to the numpy polyfit results.



$\color{DarkBlue}{\textbf{Question 7}}$

What is the largest value that you can use for the learning rate to have a stable convergence at a polynomial degree of 6?

$\color{Grey}{\textsf{<double-click and type your answer here>}}$

At higher polynomial degrees, the learning rate has to be very small because of the very large values that the higher order terms assume. This makes the polynomial very sensitive to changes in the coefficient in these terms.

To remedy this problem and improve the convergence of our algorithm, it is helpful to regard our parameterized model (i.e. the polynomial expansion) as a sum of weighted input features:

$$
y=\sum_{i=0}^{N} c_i \cdot x^i= c_0  + c_1 \cdot z_1 + c_2 \cdot z_2 + c_3 \cdot z_3 + \dots
$$

with $z_1=x$, $z_2=x^2$, $z_3=x^3$, etc.

Let's compute the values of these features, {z}, upto some maximum degree in advance:

In [ ]:
# 2D array data[(ndata, 1+maxdegree)] contains the y-labels in column 0, and the features in subsequent columns
maxdegree = 20
data = np.zeros((ntrain, 1 + maxdegree))
data[:,0] = ytrain
data[:,1] = xtrain
for i in range(2,maxdegree):
    data[:,i] = xtrain**i
# print(data)

Now, copy our steepest descent algorithm in the cell below, and implement the usage of the feature array in the forward pass and in the backward pass, instead of computing the powers of $x$.

In [ ]:

# set some parameters
ndegree = 3
learning_rate = 1e-4
maxiter = 1e6
toll = 1e-7
nprint = 10000  # print interval
ndegree += 1    # to include the last item in loops and declarations

# declare arrays
coef = np.zeros(ndegree)
grad_coef = np.zeros(ndegree)
y_pred = np.zeros(xtrain.size)

# initialize
iter = 0
converged = False
print("Fitting polynomial of degree: ",ndegree-1)
print("#  iter      loss    improvement")

# main optimization loop
while iter < maxiter and not converged:
    # Forward pass: compute y_pred = c0 + c1 x + c2 x^2 + c3 x^3 + ...
    y_pred[:] = coef[0]
    for i in range(1,ndegree):
        y_pred += coef[i] * data[:,i]

    loss_prev = loss
    # Compute loss
    loss = np.square(y_pred - ytrain).sum()

    # print progress
    if iter % nprint == 0:
        print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))
    # check convergence
    if( abs(loss - loss_prev) < toll ):
        converged = True

    # Backprop to compute gradients of coeficients with respect to loss
    grad_y_pred = 2.0 * (y_pred - ytrain)
    grad_coef[0] = grad_y_pred.sum()
    for i in range(1, ndegree):
        grad_coef[i] = (grad_y_pred * data[:,i]).sum()

    # Update weights
    coef -= learning_rate * grad_coef

    iter += 1

# print progress final epoch
print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))


# compute polynomial function and plot result together with training points and ground truth
yfit= np.zeros(xacf.size)
yfit[:] = coef[0]
for i in range(1,ndegree):
    yfit += coef[i] * xacf**i
plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.scatter(xtrain, ytrain,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='noisy data set')
plt.plot(xacf, yfit, color="red",linewidth=2.5,label='polynomial fit')
plt.legend()
plt.show()

# print polynomial expansion to screen
line = 'Result: y = ' + str('{:8.4f}'.format(coef[0]))
for i in range(1,ndegree):
    line += ' + ' + str('{:8.4f}'.format(coef[i])) + ' * x^' + str(i)
print(line)


## feature scaling

Using these recomputed features had made the algorithm a tiny bit faster. But it has not yet solved the problem of the prohibitively small learning rate required for large polynomial degrees.

To address that problem, we have to remove the very heterogeneous sensitivity to the coefficients. This can be done by scaling the feature values, such that they all span a similar range. Here, we will normalize all features using the formula:

$$ \hat{z_i} = \frac{z_i - \min(z)}{\max(z) - \min(z)} $$

in which $\hat{z_i}$ is the scaled feature value, and $\min(z)$ and $\max(z)$ are the minimum and maximum feature values respectively. This way, all features now span the range zero to one.

Let's implement in the next cell the scaling of all features in the data array. It is practical to first apply a shift by $-\min(z)$ to all data items, followed by a scaling of by $1/(\max(z) - \min(z))$. These shift and scaling factors can then be used after the optimization to recover and plot the original unscaled polynomial.

In [ ]:
# shifting and scaling of the feature array
shift = np.zeros(maxdegree)
scale = np.zeros(maxdegree)
for i in range(1,maxdegree):
#    shift[i] = - data[:,i].min()
    shift[i] = - 1.0
    scale[i]  = 1.0 / (data[:,i].max() - data[:,i].min())
    data[:,i] = (data[:,i] + shift[i]) * scale[i]
print(shift)
print(scale)

### Linear regression with scaled features

We copy once more the steepest descent algorithm algorithm and try it with the scaled data array
* how far can you push up the learning rate?
* to which polynomial degree can you now efficiently run the optimization?
* implement the "unshifting and unscaling" of the features at the end for plotting the polynomial

In [ ]:

# set some parameters
ndegree = 3
learning_rate = 1e-2
maxiter = 1e6
toll = 1e-7
nprint = 10000  # print interval
ndegree += 1    # to include the last item in loops and declarations

# declare arrays
coef = np.zeros(ndegree)
grad_coef = np.zeros(ndegree)
y_pred = np.zeros(xtrain.size)

# initialize
iter = 0
converged = False
print("Fitting polynomial of degree: ",ndegree-1)
print("#  iter      loss    improvement")

# main optimization loop
while iter < maxiter and converged == False:
    # Forward pass: compute y_pred = c0 + c1 x + c2 x^2 + c3 x^3 + ...
    y_pred[:] = coef[0]
    for i in range(1,ndegree):
        y_pred += coef[i] * data[:,i]

    loss_prev = loss
    # Compute loss
    loss = np.square(y_pred - ytrain).sum()

    # print progress
    if iter % nprint == 0:
        print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))
    # check convergence
    if( abs(loss - loss_prev) < toll ):
        converged = True

    # Backprop to compute gradients of coeficients with respect to loss
    grad_y_pred = 2.0 * (y_pred - ytrain)
    grad_coef[0] = grad_y_pred.sum()
    for i in range(1, ndegree):
        grad_coef[i] = (grad_y_pred * data[:,i]).sum()

    # Update weights
    coef -= learning_rate * grad_coef

    iter += 1

# print progress final epoch
print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))


# compute polynomial function and plot result together with training points and ground truth
yfit= np.zeros(xacf.size)
yfit[:] = coef[0]
for i in range(1,ndegree):
    yfit += coef[i] * (xacf**i + shift[i]) * scale[i]

plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.scatter(xtrain, ytrain,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='noisy data set')
plt.plot(xacf, yfit, color="red",linewidth=2.5,label='polynomial fit')
plt.ylim([-0.75, 1.2])
plt.legend()
plt.show()

# print polynomial expansion to screen
line = 'Result: y = ' + str('{:8.4f}'.format(coef[0]))
for i in range(1,ndegree):
    line += ' + ' + str('{:8.4f}'.format(coef[i])) + ' * x^' + str(i)
print(line)


## Regularization

To counteract the problem of overfitting when the problem is overdetermined, we will implement L2 regularization.
Our loss function now becomes:

$$\mathcal{L} = \frac{1}{N} \sum_{i=0}^N (y^\mathrm{pred}_i - y^\mathrm{target}_i)^2 + \lambda \sum_{j=0}^M c_j^2$$

in which $M$ is the polynomial degree and lambda is a hyperparameter to tune the dampening of large coefficient values.

We copy once more the optimization algorithm from the previous cell and implement the L2 regularization.
* investigate what the effect of regularization is on overfitting at high polynomial degrees
* investigate what is a good value for the $\lambda$ hyperparameter


In [ ]:

# set some parameters
ndegree = 16
l2scale = 1e-93
learning_rate = 1e-2
maxiter = 1e6
toll = 1e-8
nprint = 10000  # print interval
ndegree += 1    # to include the last item in loops and declarations

# declare arrays
coef = np.zeros(ndegree)
grad_coef = np.zeros(ndegree)
y_pred = np.zeros(xtrain.size)

# initialize
iter = 0
converged = False
print("Fitting polynomial of degree: ",ndegree-1)
print("#  iter      loss    improvement")

# main optimization loop
while iter < maxiter and converged == False:
    # Forward pass: compute y_pred = c0 + c1 x + c2 x^2 + c3 x^3 + ...
    y_pred[:] = coef[0]
    for i in range(1,ndegree):
        y_pred += coef[i] * data[:,i]

    loss_prev = loss
    # Compute loss
    loss = np.square(y_pred - ytrain).sum() + l2scale * np.square(coef).sum()

    # print progress
    if iter % nprint == 0:
        print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))
    # check convergence
    if( abs(loss - loss_prev) < toll ):
        converged = True

    # Backprop to compute gradients of coeficients with respect to loss
    grad_y_pred = 2.0 * (y_pred - ytrain)
    grad_coef[0] = grad_y_pred.sum()
    for i in range(1, ndegree):
        grad_coef[i] = (grad_y_pred * data[:,i]).sum() + l2scale * 2.0 * coef[i]

    # Update weights
    coef -= learning_rate * grad_coef

    iter += 1

# print progress final epoch
print('{:8d} {:12.8g} {:12.6e}'.format(iter, loss, abs(loss-loss_prev)))


# compute polynomial function and plot result together with training points and ground truth
yfit= np.zeros(xacf.size)
yfit[:] = coef[0]
for i in range(1,ndegree):
    yfit += coef[i] * (xacf**i + shift[i]) * scale[i]

plt.plot(xacf, yacf, 'g--',linewidth=2.5, label='acf(x)')
plt.scatter(xtrain, ytrain,edgecolors='b',facecolors='None',s=80,linewidth=2.5,label='noisy data set')
plt.plot(xacf, yfit, color="red",linewidth=2.5,label='polynomial fit')
plt.ylim([-0.75, 1.2])
plt.legend()
plt.show()

# print polynomial expansion to screen
line = f"Result: y = {coef[0]:8.4f}"
for i in range(1,ndegree):
    line += ' + ' + f"{coef[i]:8.4f}" + ' * x^' + str(i)
print(line)



$\color{DarkBlue}{\textbf{Question 8}}$

What happens when you choose a relatively large value of lambda?

$\color{Grey}{\textsf{<double-click and type your answer here>}}$

## Increasing the data set

As a final exercise, it is instructive to run again the optimization (without regularization) but for a (much) larger data set (simply increase ndata by a factor of 5 or 10 in the beginning of this notebook). Notice the effect on the problem of overfitting. Is the result as you would expect?

## Conclusions

This is the end of this tutorial on simple regression.

You have learned how perform a one-dimensional linear regression using a polynomial fit function. This was an example of a supervised learning task. A good number of important machine learning aspects were discussed, including defining the loss function, splitting the data into training and validation sets, feature scaling, underfitting and overfitting, learning rate, and regularization.

In case you would like to still go a bit further with this assignment topic, here are a few ideas to tryout:

1. In the following Jupyter notebook, we will use a one-layer neural network as the fit function. There are of course various other expansions of basis functions possible, other than a polynomial. For our purpose, it is interesting to use a sum of weighted sigmoidal functions, which is basically the same as a single layer neural network. We will replace the polynomial for example for the expansion $y = c_0 + \sum_{i=1}^N c_i \tanh(x)$. In neural network nomenclature, $c_0$ would be called the bias and the $\tanh()$ function is the activation function.
1. OPTIONAL: Implement K-fold cross validation. Especially in the cases were the size of the data set is limiting, it can be beneficial to split the data into $k$ equal parts, in which 1 part is kept for validation and the other parts are used for training. The training and validation can then be repeated $k$ times, every time choosing a different part for the validation. Although the estimation of the validation loss may not be very accurate in the case of small data sets, the k-fold cross validation gives $k$ estimates of the loss, which allows for an estimate of the variance (and thus the standard deviation) on the estimate. In the limit of taking $k$ equal to the number of data points, the approach is refered to as "leave-one-out" cross validation.
1. OPTIONAL: replace the L2 regularization by the L1 regularization and investigate the difference of the two types of regularization on the performance of overdetermined models.
